# Emotion classification from text (RoBERTa-base)



## 1. Setup

In [ ]:
# Import libraries
import os
import re
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# HuggingFace libraries
# - datasets:                           load the emotion dataset from the Hub
# - AutoTokenizer:                      BPE tokenizer for RoBERTa
# - AutoModelForSequenceClassification: RoBERTa encoder + classification head
# - get_linear_schedule_with_warmup:    LR scheduler standard for BERT fine-tuning
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    f1_score,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# Load dataset
raw = load_dataset("mteb/emotion")

## 2. EDA


In [ ]:
train_df = raw["train"].to_pandas()
val_df   = raw["validation"].to_pandas()
test_df  = raw["test"].to_pandas()

print("Columns:", train_df.columns.tolist())
print(f"\nRows — train: {len(train_df)} | val: {len(val_df)} | test: {len(test_df)}")
print("\nFirst rows (train):")
display(train_df.head(3))

In [ ]:
# --- Class distribution ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (df, title) in zip(
    axes,
    [(train_df, "Train"), (val_df, "Validation"), (test_df, "Test")],
):
    counts = df["label_text"].value_counts()
    ax.bar(counts.index, counts.values)
    ax.set_title(f"{title} ({len(df)} samples)")
    ax.set_xlabel("Emotion")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=30)
plt.suptitle("Class distribution per split")
plt.tight_layout()
plt.show()

# --- Text length distribution ---
train_df["word_count"] = train_df["text"].map(lambda s: len(s.split()))
print("\nWord count stats (train):")
print(train_df["word_count"].describe().round(1))

plt.figure(figsize=(7, 3))
plt.hist(train_df["word_count"], bins=40, edgecolor="white")
plt.title("Word count distribution (train)")
plt.xlabel("Words per sentence")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

## 3. Tokenizer

Unlike RNN/LSTM models that require a hand-crafted vocabulary and manual preprocessing
(lowercasing, URL removal, etc.), RoBERTa uses a **Byte-Pair Encoding (BPE)** tokenizer
trained jointly with the model on raw text.

Applying custom preprocessing (e.g. forcing lowercase) would shift the text distribution
away from what the model saw during pre-training and could hurt performance.
We therefore feed the **original text** directly into the tokenizer and let it handle
normalisation internally.

In [ ]:
# Model identifier on the HuggingFace Hub
MODEL_NAME = "roberta-base"

# max_length: cap sequences at 128 BPE tokens.
# Dataset sentences average ~13 words; at ~1.3 BPE tokens/word that is ~17 tokens.
# The 95th-percentile word count is 41 words -> ~53 BPE tokens; 128 gives ample headroom
# while staying well below the model's 512-token limit.
MAX_LEN = 128

# Load the pre-trained BPE tokenizer for RoBERTa-base
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer class :", type(tokenizer).__name__)
print("Vocab size       :", tokenizer.vocab_size)
print("Max model length :", tokenizer.model_max_length)

In [ ]:
# Preview tokenisation on a sample sentence to understand the output format.
# This mirrors the preprocess() demo in the BiLSTM notebook.
sample_text = train_df["text"].iloc[69]

encoding = tokenizer(
    sample_text,
    max_length=MAX_LEN,
    padding="max_length",
    truncation=True,
    return_tensors="pt",
)

print("Original text  :", sample_text)
print()
print("BPE tokens     :", tokenizer.convert_ids_to_tokens(encoding["input_ids"][0]))
print()
print("input_ids      :", encoding["input_ids"][0].tolist())
print()
print("attention_mask :", encoding["attention_mask"][0].tolist())
print()
# RoBERTa does NOT use token_type_ids (the NSP pre-training objective was removed)
print("token_type_ids : not used by RoBERTa (NSP objective removed)")
print()
print("Sequence length after padding/truncation:", encoding["input_ids"].shape[1])

## 4. Label preparation

In [ ]:
labels_sorted = sorted(train_df["label"].unique())
assert labels_sorted == list(range(len(labels_sorted))), "Labels should be 0..K-1"
num_classes = len(labels_sorted)
id2label = train_df.drop_duplicates("label").set_index("label")["label_text"].to_dict()
label2id = {v: k for k, v in id2label.items()}
print("num_classes:", num_classes)
print("id2label   :", id2label)

## 5. Split dataset


In [ ]:
# We use the raw text column directly.
# RoBERTa's tokenizer handles all normalisation internally,
# so no custom preprocess() step is needed here.
train_texts = train_df["text"].tolist()
val_texts   = val_df["text"].tolist()
test_texts  = test_df["text"].tolist()

y_train = train_df["label"].to_numpy()
y_val   = val_df["label"].to_numpy()
y_test  = test_df["label"].to_numpy()

print("Split sizes:", len(train_texts), len(val_texts), len(test_texts))

## 6. Tokenisation & encoding

Instead of building a word-level vocabulary and integer-encoding sequences manually
(as in the BiLSTM notebook), we delegate the entire tokenisation pipeline to the
HuggingFace tokenizer:

1. **BPE** splits each sentence into subword units.
2. Special tokens `<s>` (CLS) and `</s>` (SEP) are added automatically.
3. Sequences shorter than `MAX_LEN` are padded with `<pad>` tokens.
4. Sequences longer than `MAX_LEN` are truncated.
5. An `attention_mask` is returned: **1** for real tokens, **0** for padding positions.

All splits are encoded up-front (batch mode) to avoid repeated tokenisation
inside the training loop.

In [ ]:
def batch_encode(texts, tokenizer, max_len):
    """
    Tokenise a list of texts and return padded / truncated encodings.

    Args:
        texts    : list of raw text strings
        tokenizer: HuggingFace tokenizer instance
        max_len  : maximum sequence length (sequences are padded or truncated to this)

    Returns:
        dict with keys:
            'input_ids'      : LongTensor of shape (N, max_len) -- BPE token ids
            'attention_mask' : LongTensor of shape (N, max_len) -- 1=real, 0=padding

        Note: 'token_type_ids' is intentionally omitted because RoBERTa
        does not use the NSP objective and does not need that field.
    """
    enc = tokenizer(
        texts,
        max_length=max_len,
        padding="max_length",        # pad all sequences to exactly max_len
        truncation=True,             # truncate sequences longer than max_len
        return_tensors="pt",         # return PyTorch tensors directly
        return_token_type_ids=False, # not needed for RoBERTa
    )
    return {
        "input_ids":      enc["input_ids"],       # (N, max_len)
        "attention_mask": enc["attention_mask"],   # (N, max_len)
    }


print("Encoding train split...")
enc_train = batch_encode(train_texts, tokenizer, MAX_LEN)

print("Encoding validation split...")
enc_val = batch_encode(val_texts, tokenizer, MAX_LEN)

print("Encoding test split...")
enc_test = batch_encode(test_texts, tokenizer, MAX_LEN)

print()
print("Shapes:")
print("  input_ids  train:", enc_train["input_ids"].shape)
print("  input_ids  val  :", enc_val["input_ids"].shape)
print("  input_ids  test :", enc_test["input_ids"].shape)

## 7. PyTorch Dataset / DataLoader

In [ ]:
class EmotionDataset(Dataset):
    """
    Wraps pre-computed tokeniser encodings and integer labels into a PyTorch Dataset.

    Compared to the BiLSTM version (which returned a single tensor of token ids),
    each sample here returns three tensors:
        input_ids      : BPE token ids                shape (max_len,)
        attention_mask : 1 for real token, 0 for pad  shape (max_len,)
        label          : integer class index           shape ()

    The attention_mask is essential for RoBERTa: it tells the self-attention
    layers to ignore padding positions when computing attention weights.
    """
    def __init__(self, encodings, labels):
        # encodings: dict with 'input_ids' and 'attention_mask' tensors
        self.input_ids      = encodings["input_ids"]
        self.attention_mask = encodings["attention_mask"]
        self.labels         = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return (
            self.input_ids[i],       # (max_len,)
            self.attention_mask[i],  # (max_len,)
            self.labels[i],          # scalar
        )


def make_loader(encodings, labels, batch_size, shuffle):
    ds = EmotionDataset(encodings, labels)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


# Batch size of 32 balances GPU memory usage and training stability.
# RoBERTa-base (125M params) fits comfortably on a Colab T4 (15 GB VRAM) at this size.
BATCH_SIZE = 32

train_loader = make_loader(enc_train, y_train, BATCH_SIZE, shuffle=True)
val_loader   = make_loader(enc_val,   y_val,   BATCH_SIZE, shuffle=False)
test_loader  = make_loader(enc_test,  y_test,  BATCH_SIZE, shuffle=False)

## 8. RoBERTa-base model

`AutoModelForSequenceClassification` wraps the RoBERTa-base encoder with a
linear classification head on top of the `<s>` (CLS) token representation:

```
Input text
    ↓  BPE tokenizer
input_ids + attention_mask
    ↓  RoBERTa-base encoder (12 layers, 768 hidden, 12 attention heads)
hidden states (B, max_len, 768)
    ↓  pool <s> token -> (B, 768)
    ↓  Dropout + Linear(768, num_classes)
logits (B, 6)
```

**Key differences from BiLSTM and key advantage over DeBERTa-v3:**

| Aspect | BiLSTM | RoBERTa-base |
|---|---|---|
| Processing | Sequential (left→right + right→left) | Parallel (all positions at once) |
| Context | Fixed-window hidden state | Full self-attention over all tokens |
| Pre-training | None (trained from scratch) | Optimised BERT-style MLM on 160 GB text |
| Tokenizer | Word-level vocab (~15K) | BPE subword vocab (50K) |
| Parameters | ~1M | ~125M |
| Stability | High | High (no special tricks needed) |

In [ ]:
def count_params(m):
    # Count number of parameters that will actually be updated during training
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


# AutoModelForSequenceClassification automatically attaches a classification head
# (dropout + linear layer) on top of the pooled <s> (CLS) token representation.
#
# id2label / label2id are stored in model.config so that
# model.config.id2label works correctly at inference time.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id,
).to(device)

print("Model            :", type(model).__name__)
print("Trainable params :", f"{count_params(model):,}")

In [ ]:
# Quick sanity check: one forward pass to verify output shapes
input_ids_demo, mask_demo, _ = next(iter(train_loader))
with torch.no_grad():
    out = model(
        input_ids=input_ids_demo.to(device),
        attention_mask=mask_demo.to(device),
    )

# AutoModelForSequenceClassification returns a SequenceClassifierOutput dataclass.
# .logits contains the raw (un-normalised) class scores.
print("Output type  :", type(out).__name__)
print("Logits shape :", out.logits.shape)  # expected: (BATCH_SIZE, num_classes)

## 9. Training — CrossEntropyLoss, AdamW, linear warmup, early stopping on val macro-F1

Fine-tuning a pre-trained transformer requires a different optimisation recipe
compared to training a BiLSTM from scratch:

* **AdamW** instead of Adam: decouples weight decay from the gradient update,
  which is mathematically more correct and is standard for BERT-family models.
* **Very small learning rate (2e-5)**: the pre-trained weights are already excellent;
  a large LR would destroy the learned representations ("catastrophic forgetting").
* **Selective weight decay**: bias terms and LayerNorm weights are excluded from
  regularisation (standard practice for transformer fine-tuning).
* **Linear warmup scheduler**: LR ramps from 0 → peak over the first 10% of training
  steps, then linearly decays to 0. This stabilises early gradient updates.
* **Fewer epochs (5 max)**: transformers converge quickly on fine-tuning tasks.
* **Early stopping** on val macro-F1 with patience=2.

In [ ]:
# Train the model for one epoch
def train_one_epoch(model, loader, optimizer, scheduler, criterion):
    """
    Train the model for one epoch.

    Compared to the BiLSTM version, each batch now unpacks three tensors
    (input_ids, attention_mask, labels) and the scheduler is stepped once
    per batch (not per epoch), which is the standard transformer fine-tuning convention.

    Args:
        model     : RoBERTa classification model
        loader    : DataLoader for the training split
        optimizer : AdamW optimizer
        scheduler : linear warmup/decay scheduler
        criterion : CrossEntropyLoss

    Returns:
        avg_loss  : average training loss over all samples in the epoch
    """
    # Set model to training mode
    # This enables dropout and other training-specific behaviour
    model.train()
    total_loss = 0.0

    # Loop through batches from DataLoader
    for input_ids, attention_mask, labels in loader:
        input_ids      = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels         = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        # Forward pass: model returns a SequenceClassifierOutput dataclass
        # We extract .logits (shape: B x num_classes) and compute loss
        # manually to keep the loop consistent with the BiLSTM notebook.
        output = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = output.logits  # (B, num_classes)

        # Compute loss between predicted logits and true labels
        loss = criterion(logits, labels)

        # Backpropagation: compute gradients
        loss.backward()

        # Gradient clipping helps stabilise transformer fine-tuning
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Update model parameters
        optimizer.step()

        # Step the LR scheduler once per batch (transformer convention)
        scheduler.step()

        # Accumulate total loss weighted by batch size
        total_loss += loss.item() * input_ids.size(0)

    # Return average loss over the whole training set
    return total_loss / len(loader.dataset)


# Evaluate model on validation or test set
@torch.no_grad()
def evaluate(model, loader, criterion=None):
    """
    Evaluate model on a validation or test DataLoader.

    Args:
        model     : RoBERTa classification model
        loader    : DataLoader for the target split
        criterion : CrossEntropyLoss (optional; loss is skipped if None)

    Returns:
        metrics : dict with 'acc', 'macro_f1', 'weighted_f1', and optionally 'loss'
        y_true  : numpy array of ground-truth labels
        y_pred  : numpy array of predicted labels
    """
    # Set model to evaluation mode - disables dropout
    model.eval()
    total_loss = 0.0
    all_y, all_p = [], []

    for input_ids, attention_mask, labels in loader:
        input_ids      = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels         = labels.to(device)

        # Forward pass
        output = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = output.logits  # (B, num_classes)

        # Compute loss only if criterion is provided
        if criterion is not None:
            total_loss += criterion(logits, labels).item() * input_ids.size(0)

        # Predicted class = index of highest logit
        all_p.append(logits.argmax(dim=-1).cpu().numpy())

        # Store true labels
        all_y.append(labels.cpu().numpy())

    # Merge all batches into full arrays
    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_p)

    # Compute evaluation metrics
    metrics = {
        "acc":         accuracy_score(y_true, y_pred),
        "macro_f1":    f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    }

    # Add average loss if criterion was used
    if criterion is not None:
        metrics["loss"] = total_loss / len(loader.dataset)

    return metrics, y_true, y_pred


# Training loop flow:
# 1. Compute total training steps and warmup steps from epochs & loader length
# 2. Build AdamW optimizer with two parameter groups:
#    - weight decay applied to all parameters EXCEPT bias and LayerNorm weights
#    - no weight decay on bias / LayerNorm (standard transformer practice)
# 3. Build linear warmup scheduler:
#    - LR ramps from 0 to peak over warmup_steps
#    - then linearly decays to 0 over the remaining steps
# 4. Train for each epoch:
#    - train_one_epoch: forward, loss, backward, clip, optimizer step, scheduler step
#    - evaluate on val_loader: acc, macro-F1, weighted-F1, loss
#    - save metrics into history
#    - check if val macro-F1 improved
#      improved     -> save best model state, reset stale counter
#      not improved -> increment stale counter
#    - early stop if stale >= patience
# 5. Restore best model weights before returning
def train_model(
    model,
    train_loader,
    val_loader,
    epochs=5,
    lr=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,  # fraction of total steps used for linear LR warmup
    patience=2,        # stop if val macro-F1 does not improve for this many epochs
):
    criterion = nn.CrossEntropyLoss()

    # --- Optimizer: AdamW with selective weight decay ---
    # Do NOT apply weight decay to bias terms or LayerNorm weights.
    # These parameters have no "magnitude" to regularise and decaying them
    # hurts performance (standard practice for BERT-family fine-tuning).
    no_decay = {"bias", "LayerNorm.weight"}
    param_groups = [
        {
            # All parameters whose names do NOT contain 'bias' or 'LayerNorm.weight'
            "params": [
                p for n, p in model.named_parameters()
                if not any(nd in n for nd in no_decay)
            ],
            "weight_decay": weight_decay,
        },
        {
            # Bias and LayerNorm parameters: no weight decay
            "params": [
                p for n, p in model.named_parameters()
                if any(nd in n for nd in no_decay)
            ],
            "weight_decay": 0.0,
        },
    ]
    optimizer = torch.optim.AdamW(param_groups, lr=lr)

    # --- Scheduler: linear warmup then linear decay ---
    total_steps  = len(train_loader) * epochs
    warmup_steps = int(total_steps * warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    print(f"Total steps: {total_steps} | Warmup steps: {warmup_steps}")

    # Store training history for plotting later
    history  = {"train_loss": [], "val_loss": [], "val_macro_f1": [], "val_acc": []}

    # Track best model according to validation macro-F1
    best_state = None
    best_f1    = -1.0
    stale      = 0  # number of epochs without improvement

    for ep in range(1, epochs + 1):
        # Train one epoch
        tl = train_one_epoch(model, train_loader, optimizer, scheduler, criterion)

        # Evaluate on validation set
        vm, _, _ = evaluate(model, val_loader, criterion)

        # Save current metrics
        history["train_loss"].append(tl)
        history["val_loss"].append(vm["loss"])
        history["val_macro_f1"].append(vm["macro_f1"])
        history["val_acc"].append(vm["acc"])

        # Check whether validation macro-F1 improved
        if vm["macro_f1"] > best_f1 + 1e-5:
            best_f1 = vm["macro_f1"]

            # Save a copy of the best model weights (moved to CPU to free GPU memory)
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1

        # Print progress every epoch
        if ep == 1 or ep % 1 == 0 or stale == 0:
            print(
                f"Epoch {ep:02d} "
                f"train_loss={tl:.4f} "
                f"val_loss={vm['loss']:.4f} "
                f"val_acc={vm['acc']:.4f} "
                f"val_macro_f1={vm['macro_f1']:.4f}"
            )

        # Stop early if no improvement for 'patience' epochs
        if stale >= patience:
            print(
                f"Early stopping at epoch {ep} "
                f"(no val macro-F1 improvement for {patience} epochs)."
            )
            break

    # Restore best model weights before returning
    if best_state is not None:
        model.load_state_dict(best_state)

    return history, best_f1


# Plot training curves
def plot_history(history, title="Training"):
    fig, ax = plt.subplots(1, 2, figsize=(9, 3))

    # Plot train/validation loss
    ax[0].plot(history["train_loss"], label="train")
    ax[0].plot(history["val_loss"],   label="val")
    ax[0].set_title("Loss")
    ax[0].legend()

    # Plot validation metrics
    ax[1].plot(history["val_macro_f1"], label="val macro-F1")
    ax[1].plot(history["val_acc"],      label="val acc", alpha=0.6)
    ax[1].set_title("Val metrics")
    ax[1].legend()

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

## 10. Train RoBERTa-base

In [ ]:
# Re-initialise model to ensure clean weights before training.
# This is important if you re-run this cell: it avoids resuming from a partially
# trained checkpoint and guarantees a reproducible starting point.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id,
).to(device)

history, best_val_f1 = train_model(
    model,
    train_loader,
    val_loader,
    epochs=5,
    lr=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    patience=2,
)

print(f"\nBest val macro-F1 (main): {best_val_f1}")
plot_history(history, title="RoBERTa-base")

## 11. Evaluation

In [ ]:
def full_report(y_true, y_pred, split_name="val"):
    """
    Print a full classification report and plot the confusion matrix.

    Args:
        y_true     : true labels
        y_pred     : predicted labels
        split_name : name of the data split, e.g. 'validation' or 'test'

    Returns:
        cm: confusion matrix as a NumPy array
    """

    # Create class names in label-id order
    names = [id2label[i] for i in sorted(id2label.keys())]

    # Print split name
    print(f"=== {split_name} ===")

    # Print overall accuracy
    print("Accuracy:", accuracy_score(y_true, y_pred))

    # Compute per-class precision, recall, and F1
    # average=None means return metric for each class separately
    p, r, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average=None,
        labels=sorted(id2label.keys()),
    )

    # Assemble per-class metrics into a DataFrame for readability
    metrics_df = pd.DataFrame(
        {"precision": p, "recall": r, "f1": f1},
        index=names,
    ).round(4)
    print("\nPer-class metrics:")
    display(metrics_df)

    # Print macro and weighted averages
    print("\nMacro    F1:", round(f1_score(y_true, y_pred, average="macro"),    4))
    print("Weighted F1:", round(f1_score(y_true, y_pred, average="weighted"), 4))

    # Full sklearn classification report
    print("\nFull classification report:")
    print(classification_report(y_true, y_pred, target_names=names))

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=sorted(id2label.keys()))

    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=names,
        yticklabels=names,
        cmap="Blues",
    )
    plt.title(f"Confusion Matrix — {split_name}")
    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.show()

    return cm


# Evaluate the trained model on the validation set
_, yv_t, yv_p = evaluate(model, val_loader, nn.CrossEntropyLoss())
full_report(yv_t, yv_p, "Validation RoBERTa-base")

In [ ]:
# Evaluate the trained model on the held-out test set
_, yt_t, yt_p = evaluate(model, test_loader, nn.CrossEntropyLoss())
full_report(yt_t, yt_p, "Test RoBERTa-base")

## 12. Error analysis


In [ ]:
# Run evaluation on the validation set
_, yv_t, yv_p = evaluate(model, val_loader, None)

# Create ordered class names from label ids
names = [id2label[i] for i in sorted(id2label.keys())]

# Build confusion matrix
cm = confusion_matrix(yv_t, yv_p, labels=sorted(id2label.keys()))

# Find the most common confusion pairs
pairs = []
n = cm.shape[0]
for i in range(n):
    for j in range(n):
        if i != j and cm[i, j] > 0:
            pairs.append((cm[i, j], names[i], names[j]))
pairs.sort(reverse=True)

# Print the most frequent confusion types
print("Top confused (true -> pred):")
for c, a, b in pairs[:8]:
    print(f"  {c:4d}  {a} -> {b}")

# Show some actual misclassified validation examples
val_texts_arr = val_df["text"].to_numpy()
wrong_idx = np.where(yv_t != yv_p)[0]
sample = np.random.default_rng(SEED).choice(wrong_idx, size=min(12, len(wrong_idx)), replace=False)
print("\nSample errors (val):")
for i in sample:
    print(f"\n true={names[yv_t[i]]} pred={names[yv_p[i]]}")
    print(" ", val_texts_arr[i][:280])